In [ ]:
import boto3
import os
from boto3.s3.transfer import TransferConfig
import threading
import sys
import json

access_key_id = 'xyz'
secret_access_key = 'xyz'
LOCAL_S3_PROXY_SERVICE_URL = 'https://xyz.xyz.com'

s3 = boto3.client('s3',
    aws_access_key_id=access_key_id,
    aws_secret_access_key=secret_access_key,
    endpoint_url=LOCAL_S3_PROXY_SERVICE_URL
)

In [ ]:
response = s3.create_bucket(
    Bucket='bucket2'
)

In [ ]:
s3.list_buckets()['Buckets']

In [ ]:
s3.list_objects_v2(Bucket='bucket2')['Contents']

In [ ]:
config = TransferConfig(
    multipart_threshold=1024 * 25,  # 25MB
    max_concurrency=10,
    multipart_chunksize=1024 * 25,
    use_threads=True
)

class ProgressPercentage:
    def __init__(self, filename):
        self._filename = filename
        self._size = float(os.path.getsize(filename))
        self._seen_so_far = 0
        self._lock = threading.Lock()

    def __call__(self, bytes_amount):
        with self._lock:
            self._seen_so_far += bytes_amount
            percentage = (self._seen_so_far / self._size) * 100
            sys.stdout.write(
                f"\r{self._filename}: {self._seen_so_far} / {self._size} "
                f"({percentage:.2f}%)"
            )
            sys.stdout.flush()

# Upload with progress
s3.upload_file(
    '/Users/xyz/Downloads/PS2_BIOS.zip',
    'bucket2',
    'PS2_BIOS.zip',
    Config=config,
    Callback=ProgressPercentage('/Users/xyz/Downloads/PS2_BIOS.zip')
)

In [ ]:
s3.upload_file(
    '/Users/xyz/Downloads/rockyou.txt',
    'hello',
    'rockyou.txt'
)

s3.upload_file(
    '/Users/xyz/Downloads/Digital datatest.xlsx',
    'hello',
    '/xlsx/Digital datatest.xlsx'
)

In [ ]:
s3.download_file(
    'hello',
    '/xlsx/Digital datatest.xlsx',
    '/Users/xyz/Downloads/Digital datatest5.xlsx',
)

In [ ]:
data = {
  'message': 'Hello world, xyz!!',
  'created_at': '2020-06-03 05:36:00'
}
formatted_data = json.dumps(data).encode("UTF-8")# encode utf-8 bytes

In [ ]:
s3.put_object(
    Body=formatted_data
    , Bucket='hello'
    , Key='json/test'
)

In [ ]:
response = s3.get_object(
    Bucket='hello',
    Key='json/test')

In [ ]:
response

In [ ]:
val = response['Body'].read()

In [ ]:
val

In [ ]:
formatted_data = json.loads(val.decode('utf-8'))

In [ ]:
formatted_data

In [ ]:
formatted_data['message']